In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Simulated dataset (text embeddings + sentiment labels)
# Let's assume we already have sentence embeddings (e.g., from BERT)
np.random.seed(42)
X = np.random.randn(100, 300)  # 100 samples, 300-dim embeddings
y = np.random.choice([0, 1, 2], size=100)  # 0=neg, 1=neutral, 2=pos

# Simple policy network
class PolicyNet(nn.Module):
    def __init__(self, input_dim=300, num_classes=3):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return torch.softmax(self.fc(x), dim=-1)

policy = PolicyNet()
optimizer = optim.Adam(policy.parameters(), lr=0.01)

# Reinforcement Learning training loop
for epoch in range(200):
    epoch_reward = 0
    for i in range(len(X)):
        x = torch.tensor(X[i], dtype=torch.float32)
        probs = policy(x)
        action = torch.multinomial(probs, 1).item()  # sample sentiment

        # Reward: +1 if correct, -1 if wrong
        reward = 1 if action == y[i] else -1
        epoch_reward += reward

        # Compute log-prob and policy gradient loss
        log_prob = torch.log(probs[action])
        loss = -log_prob * reward  # REINFORCE objective

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}, Avg Reward: {epoch_reward / len(X):.3f}")


Epoch 20, Avg Reward: 0.780
Epoch 40, Avg Reward: 0.780
Epoch 60, Avg Reward: 0.780
Epoch 80, Avg Reward: 0.720
Epoch 100, Avg Reward: 0.720
Epoch 120, Avg Reward: 0.720
Epoch 140, Avg Reward: 0.760
Epoch 160, Avg Reward: 0.760
Epoch 180, Avg Reward: 0.760
Epoch 200, Avg Reward: 0.760
